In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchsummary import summary
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
import time

/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `t

In [2]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size ,patch_size ,num_hiddens):
        super().__init__()
        self.num_patches= (img_size// patch_size ) ** 2
        self.conv = nn.Conv2d(3,num_hiddens,kernel_size=patch_size,stride=patch_size)

    def forward(self,x):
        return self.conv(x).flatten(2).transpose(1,2)
    

class ViTMLP(nn.Module):
    def __init__(self,mlp_num_hidden, mlp_num_outputs,dropout=0.5):
        super().__init__()
        self.dense1=nn.LazyLinear(mlp_num_hidden)
        self.gelu = nn.GELU()
        self.dropout1=nn.Dropout(dropout)
        self.dense2=nn.LazyLinear(mlp_num_outputs)
        self.dropout2=nn.Dropout(dropout)

    def forward(self,x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(self.dense1(x)))))
    

class ViTBlock(nn.Module):
    def __init__(self,num_hidden,norm_shape,mlp_num_hiddens,num_heads,dropout,use_bias=False):
        super().__init__()
        self.ln1= nn.LayerNorm(norm_shape)
        self.attention = nn.MultiheadAttention(num_hidden,num_heads,dropout,use_bias,batch_first=True)
        self.ln2= nn.LayerNorm(norm_shape)
        self.mlp= ViTMLP(mlp_num_hiddens,num_hidden,dropout)

    def forward(self,x):
        x2= self.ln1(x)
        attention_output,_=self.attention(x2,x2,x2)
        x = x + attention_output
        x2= self.ln2(x)
        mlp_output= self.mlp(x2)
        x = x + mlp_output
        return x 
    


In [3]:
class ViT(nn.Module):
    def __init__(self, eta, n_iter, random_state, batch_size, img_size, patch_size,
                 num_hidden, norm_shape, mlp_num_hiddens, num_heads, num_blks,
                 num_classes=100, dropout=0.2):
        super().__init__()
        self.eta = eta
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.random_state = random_state

        self.patch_embedding = PatchEmbedding(img_size, patch_size, num_hidden)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hidden))
        num_steps = self.patch_embedding.num_patches + 1
        self.pos_embedding = nn.Parameter(torch.randn(1, num_steps, num_hidden))
        self.drop_out = nn.Dropout(dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(num_hidden, norm_shape, mlp_num_hiddens,
                                                    num_heads, dropout, use_bias=False))
        self.head = nn.Sequential(nn.LayerNorm(num_hidden), nn.Linear(num_hidden, num_classes))

        self.device = torch.device(
            "cuda" if torch.cuda.is_available()
            else "mps" if torch.backends.mps.is_available()
            else "cpu"
        )
        print(f"Using device: {self.device}")

        self.train_losses_     = []
        self.train_accuracies_ = []
        self.val_losses_       = []
        self.val_accuracies_   = []
        self.to(self.device)

    def forward(self, x):
        x = self.patch_embedding(x)
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), 1)
        x = self.drop_out(x + self.pos_embedding)
        for blk in self.blks:
            x = blk(x)
        x = self.head(x[:, 0])
        return x
    
    def iter_mini_batch(self,X,y):
        dataset=TensorDataset(X,y)
        dataloader= DataLoader(dataset,batch_size=self.batch_size,shuffle=True)
        return dataloader
    
    def fit(self, X,y,X_test,y_test):
        torch.manual_seed(self.random_state)
        start_time = time.time()
        optimizer = optim.SGD(self.parameters,lr=self.eta,weight_decay=0.001)
        criterion= nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.n_iter)
        for epoch in range(self.n_iter):
            epoch_start= time.time()
            self.train()
            train_loader = self.iter_mini_batch(X,y)
            running_loss=0.0
            correct=0
            total =0

            for X_batch,y_batch in train_loader:
                X_batch= X_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                optimizer.zero_grad()
                y_hat= self(X_batch)
                loss= criterion(y_hat,y)
                loss.backward()
                optimizer.step()

                running_loss+=loss.item()
                preds=y_hat.argmax(dim=1)
                correct+= (preds == y_batch).sum().item()
                total += y_batch.numel()
            epoch_time = time.time()-epoch_start
            epoch_loss = running_loss/(len(train_loader))
            epoch_acc= 100 * correct/total

            self.train_losses_.append(epoch_loss)
            self.train_accuracies_.append(epoch_acc)

            val_loss, val_acc = self._evaluate(X_test, y_test, criterion)
            self.val_losses_.append(val_loss)
            self.val_accuracies_.append(val_acc)
            current_lr = optimizer.param_groups[0]['lr']

            print(f'Epoch {epoch+1:>3}/{self.n_iter} | '
                  f'Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}% | '
                  f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% |'
                  f'lr: {current_lr:.6f} |'
                  f'Time: {epoch_time:.1f}s')

        self.training_time_ = time.time() - start_time    #stop the clock
        mins = self.training_time_ // 60
        secs = self.training_time_ % 60
        print(f"\nTraining complete: {int(mins)}m {secs:.1f}s")
        return self
    
    def _evaluate(self, X_test, y_test, criterion):
        test_loader=self.iter_mini_batch(X_test,y_test)
        self.eval()
        running_loss = 0.0
        correct      = 0
        total        = 0
        with torch.no_grad():
            for xin, target in test_loader:
                xin = xin.to(self.device)
                target = target.to(self.device)

                outputs = self(xin)
                loss=criterion(outputs.view(-1, outputs.size(-1)), target.view(-1))

                running_loss+= loss.item()
                predicted = outputs.argmax(dim=-1)
                total += target.numel()
                correct += (predicted == target).sum().item()
        val_loss = running_loss / len(test_loader)
        val_acc  = 100.0 * correct / total
        return val_loss, val_acc
    
    def predict(self, X):
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)
        self.eval()
        with torch.no_grad():
            X        = X.to(self.device)
            logits   = self(X)
            _, preds = torch.max(logits, dim=1)
        return preds.cpu().numpy()
    def accuracy(self, X, y):
        if isinstance(y, torch.Tensor):
            y = y.numpy()
        preds = self.predict(X)
        return 100.0 * np.mean(preds == y)
    def training_summary(self):
      mins = int(self.training_time_ // 60)
      secs = self.training_time_ % 60
      params = sum(p.numel() for p in self.parameters())
      print(f"Parameters:    {params:,}")
      print(f"Training time: {mins}m {secs:.1f}s")
      print(f"Best val acc:  {max(self.val_accuracies_):.2f}%")
      print(f"Final test acc: call model.accuracy(X_test, y_test)")
            

        